# Classification → Regression: Submission Skeleton (โมเดลเดียวต่อ stage)

**วิธีใช้ตอนสอบ**
1. เทียบโมเดล/จูนใน `classification_then_regression.ipynb` จนรู้ว่า classifier และ regressor ตัวไหนชนะ พร้อม best params
2. เปิดไฟล์นี้ แก้ `CLASS_COL`, `REG_COL` (และ `DROP_COLS`) ใน cell Config
3. แก้ **SLOT A** (classifier) และ **SLOT B** (regressor) ให้เป็นโมเดลและ params ที่เลือก (แก้แค่ใน `make_classifier()` / `make_regressor()` ข้างละ 1 บรรทัด)
4. Run all → ได้ผล Stage 1 (accuracy/F1), Stage 2 (R²/RMSE/MAE) และ `predict_all(df_new)`

Stage 2 ใช้ out-of-fold prediction ของ Stage 1 เป็น feature (กัน leakage) ไม่ต้องแก้อะไรในส่วนนี้

## 0. โหลดข้อมูล

In [ ]:
import pandas as pd

# แบบ 1: ไฟล์ csv อยู่ใน Colab แล้ว
df = pd.read_csv('data.csv')

# แบบ 2: ต้องอัปโหลดเอง
# from google.colab import files
# files.upload()
# df = pd.read_csv('ชื่อไฟล์.csv')

# แบบ 3: อาจารย์ให้ dataframe มาแล้ว (เช่นชื่อ data) -> df = data.copy()

df.head()

## 1. Config

In [ ]:
import warnings
warnings.filterwarnings('ignore')

CLASS_COL = 'ชื่อคอลัมน์class'    # <-- แก้ตรงนี้ (คอลัมน์ class)
REG_COL   = 'ชื่อคอลัมน์y'      # <-- แก้ตรงนี้ (คอลัมน์ y ต่อเนื่อง)
DROP_COLS = []               # เช่น ['id', 'name']

TEST_SIZE = 0.2
SEED      = 42
N_SPLITS  = 5
MAX_NULL_COL  = 0.50
ADD_INDICATOR = 0.05

USE_LOG_TARGET   = False     # True = regress บน log1p(y)
ADD_PROBA        = True      # ส่ง probability ของแต่ละ class ไป Stage 2
ADD_CLASS_ONEHOT = True      # ส่ง class ที่ทำนาย (one-hot) ไป Stage 2

## 2. Data prep + split (ไม่ต้องแก้)

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

data = df.drop_duplicates().copy()
data = data.dropna(subset=[CLASS_COL, REG_COL])
data = data.drop(columns=[c for c in DROP_COLS if c in data.columns])

feat = [c for c in data.columns if c not in (CLASS_COL, REG_COL)]
ratio = data[feat].isna().mean()
feat = [c for c in feat if ratio[c] <= MAX_NULL_COL]
ind_cols = [c for c in feat if data[c].isna().mean() > ADD_INDICATOR]
raw_feat = feat.copy()
cat_cols = data[feat].select_dtypes(exclude='number').columns.tolist()
num_cols = [c for c in feat if c not in cat_cols] + [c + '_missing' for c in ind_cols]

def clean_X(d_):
    X_ = d_[raw_feat].copy()
    for c in ind_cols:
        X_[c + '_missing'] = X_[c].isna().astype(int)
    for c in cat_cols:
        X_[c] = X_[c].astype(object).fillna('missing').astype(str)
    return X_

def make_prep():
    return ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)])

X_all = clean_X(data)
le = LabelEncoder()
yc_all = le.fit_transform(data[CLASS_COL])
class_names = [str(c) for c in le.classes_]
yr_all = data[REG_COL].to_numpy(float)
K = len(class_names)

try:
    idx_tr, idx_te = train_test_split(np.arange(len(X_all)), test_size=TEST_SIZE, random_state=SEED, stratify=yc_all)
except ValueError:
    idx_tr, idx_te = train_test_split(np.arange(len(X_all)), test_size=TEST_SIZE, random_state=SEED)

X_train, X_test = X_all.iloc[idx_tr], X_all.iloc[idx_te]
yc_train, yc_test = yc_all[idx_tr], yc_all[idx_te]
yr_train, yr_test = yr_all[idx_tr], yr_all[idx_te]
print('classes:', dict(enumerate(class_names)), '| train:', X_train.shape, ' test:', X_test.shape)

## 3. SLOT A: Classifier  ← แก้ตรง `make_classifier()` ให้เป็นโมเดลที่เลือก
แก้บรรทัด `return ...` ให้เป็นโมเดล + params ที่จูนได้ (ตัวอย่างอื่นอยู่ในคอมเมนต์)

In [ ]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

def make_classifier():
    return RandomForestClassifier(n_estimators=300, max_depth=None, min_samples_leaf=1, random_state=SEED, n_jobs=-1)   # <-- SLOT A
    # return LogisticRegression(C=10, max_iter=2000)
    # return GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=SEED)
    # return SVC(C=10, gamma='scale', probability=True, random_state=SEED)       # ต้อง probability=True
    # return KNeighborsClassifier(n_neighbors=7, weights='distance')
    # return DecisionTreeClassifier(max_depth=6, random_state=SEED)
    # return GaussianNB()
    # class ไม่สมดุล: เพิ่ม class_weight='balanced' (ใน Logistic/SVC/DecisionTree/RandomForest)

## 4. SLOT B: Regressor  ← แก้ตรง `make_regressor()`

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler

def make_regressor():
    return GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=SEED)   # <-- SLOT B
    # return Ridge(alpha=10)
    # return RandomForestRegressor(n_estimators=300, max_depth=None, min_samples_leaf=3, random_state=SEED, n_jobs=-1)
    # return Lasso(alpha=0.1, max_iter=20000)
    # return KNeighborsRegressor(n_neighbors=7, weights='distance')
    # return DecisionTreeRegressor(max_depth=6, random_state=SEED)
    # return TransformedTargetRegressor(regressor=SVR(C=10), transformer=StandardScaler())   # SVR ต้อง scale y ด้วย

def build_regressor():
    m = make_regressor()
    if USE_LOG_TARGET and yr_train.min() > -1:
        m = TransformedTargetRegressor(regressor=m, func=np.log1p, inverse_func=np.expm1)
    return m

## 5. Stage 1: train + วัดผล (accuracy / F1)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

clf_final = Pipeline([('prep', make_prep()), ('clf', make_classifier())]).fit(X_train, yc_train)
pred_c = clf_final.predict(X_test)

print(f'Accuracy    = {accuracy_score(yc_test, pred_c):.4f}')
print(f'F1 macro    = {f1_score(yc_test, pred_c, average="macro"):.4f}')
print(f'F1 weighted = {f1_score(yc_test, pred_c, average="weighted"):.4f}')
if K == 2:
    print(f'F1 (positive = "{class_names[1]}") = {f1_score(yc_test, pred_c):.4f}')
print(classification_report(yc_test, pred_c, target_names=class_names, digits=3))
ConfusionMatrixDisplay(confusion_matrix(yc_test, pred_c), display_labels=class_names).plot(cmap='Blues', colorbar=False)
plt.title('Confusion matrix (test)'); plt.show()

## 6. Stage 2: สร้าง feature จาก Stage 1 (OOF) + train + วัดผล (R² / RMSE / MAE)
train ใช้ out-of-fold prediction | test ใช้โมเดล Stage 1 ที่ fit เต็ม

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

K_CV = max(2, min(N_SPLITS, int(np.bincount(yc_train).min())))
cv_cls = StratifiedKFold(K_CV, shuffle=True, random_state=SEED)

prep_final = clf_final.named_steps['prep']
Xtr_p, Xte_p = prep_final.transform(X_train), prep_final.transform(X_test)
oof_proba = cross_val_predict(clone(clf_final), X_train, yc_train, cv=cv_cls, method='predict_proba')
te_proba  = clf_final.predict_proba(X_test)

def make_stage2(Xp, proba):
    parts = [Xp]
    if ADD_PROBA:        parts.append(proba)
    if ADD_CLASS_ONEHOT: parts.append(np.eye(K)[proba.argmax(1)])
    return np.hstack(parts)

Ztr, Zte = make_stage2(Xtr_p, oof_proba), make_stage2(Xte_p, te_proba)
reg_final = build_regressor().fit(Ztr, yr_train)

p = reg_final.predict(Zte)
print(f'R2 train = {r2_score(yr_train, reg_final.predict(Ztr)):.4f}')
print(f'TEST  R2={r2_score(yr_test, p):.4f}  RMSE={np.sqrt(mean_squared_error(yr_test, p)):.4f}  MAE={mean_absolute_error(yr_test, p):.4f}')

## 7. ทำนายแบบ end-to-end + ข้อมูลใหม่

In [ ]:
def predict_all(df_new):
    """รับ dataframe ดิบ (ไม่ต้องมีคอลัมน์ class/y) -> DataFrame: pred_class, pred_value, proba_*"""
    Xn = clean_X(df_new)
    proba = clf_final.predict_proba(Xn)
    Zn = make_stage2(clf_final.named_steps['prep'].transform(Xn), proba)
    out = pd.DataFrame({'pred_class': le.inverse_transform(proba.argmax(1)),
                        'pred_value': reg_final.predict(Zn)}, index=df_new.index)
    for i, c in enumerate(class_names):
        out['proba_' + c] = proba[:, i]
    return out

test_raw = data.iloc[idx_te]
res = predict_all(test_raw)
print(f'[Stage 1] Accuracy={accuracy_score(test_raw[CLASS_COL], res.pred_class):.4f}  F1 macro={f1_score(test_raw[CLASS_COL], res.pred_class, average="macro"):.4f}')
print(f'[Stage 2] R2={r2_score(yr_test, res.pred_value):.4f}  RMSE={np.sqrt(mean_squared_error(yr_test, res.pred_value)):.4f}  MAE={mean_absolute_error(yr_test, res.pred_value):.4f}')
res.head()

# ถ้ามีไฟล์ test ของอาจารย์:
# new_df = pd.read_csv('test.csv')
# predict_all(new_df)[['pred_class', 'pred_value']].to_csv('predictions.csv', index=False)

## 8. (ถ้าต้องส่งเป็นไฟล์โมเดล) บันทึก/โหลด

In [ ]:
import joblib

# บันทึก: เก็บทั้ง 2 stage + ตัวช่วยที่ predict_all ต้องใช้
joblib.dump({'clf': clf_final, 'reg': reg_final, 'le': le, 'class_names': class_names,
             'flags': {'ADD_PROBA': ADD_PROBA, 'ADD_CLASS_ONEHOT': ADD_CLASS_ONEHOT},
             'cols': {'raw_feat': raw_feat, 'ind_cols': ind_cols, 'cat_cols': cat_cols}},
            'cls_reg_model.joblib')
print('บันทึกแล้ว: cls_reg_model.joblib')
# โหลดกลับ: bundle = joblib.load('cls_reg_model.joblib')  (ต้องมีฟังก์ชัน clean_X / make_stage2 ในสภาพแวดล้อมที่โหลด)